<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_09-2_RPG_MCP_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 2 | AI RPG 게임 마스터: MCP로 게임 서버 만들기

**실행 환경:** Google Colab · **API 키 불필요**

> **핵심 질문:** 실습 1에서 만든 `attack_enemy()` 같은 함수를 다른 AI 앱도 공통 방식으로 발견하고 호출하려면 어떻게 해야 할까요?

이번 실습에서는 게임 함수를 **MCP Server의 Tool**로 노출하고, **Resource**(몬스터 설명)와 **Prompt**(전투 전략 템플릿)를 추가합니다. 실제 LLM은 사용하지 않습니다. **MCP 표준 통신과 도구 실행을 먼저 검증하는 수업**입니다.

### 학습 목표
1. MCP의 **Server / Client / Host** 역할을 설명할 수 있다.
2. `@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`를 구별한다.
3. `Client(mcp)`로 서버의 도구를 발견하고 정상/오류 호출을 검증한다.
4. 읽기 작업과 상태 변경 작업의 차이를 테스트한다.

## 0. Colab 실행 안내

- 위에서 아래로 순서대로 실행하세요. `런타임 → 모두 실행`을 권장합니다.
- 설치 셀은 인터넷 연결이 필요합니다. 설치 이후 기본 실습은 API 키나 외부 게임 서버가 필요하지 않습니다.
- **In-memory Client**를 사용하므로 서버 포트 열기·별도 프로세스 실행은 필요하지 않습니다.
- 실습 1과 **같은 기본 게임 규칙**을 사용하지만, 이 노트북만으로 독립 실행할 수 있도록 게임 로직을 다시 정의합니다.
- 기준 패키지: 강의자료의 `mcp==2.3.0`. 배포된 SDK 버전에 따라 API가 바뀔 수 있으므로 버전을 고정합니다.

**검증 범위 안내:** 노트북 생성 환경에서는 패키지 설치용 네트워크가 차단되어 MCP 호출을 실제로 실행하지 못했습니다. 아래 자동 테스트는 Colab에서 직접 확인하세요.

In [1]:
# Colab에는 mcp 라이브러리가 기본 설치되어 있지 않을 수 있습니다.
# 강의자료와 일치하도록 버전을 고정해 환경 차이를 줄입니다.
%pip -q install "mcp==2.3.0"

In [2]:
# 설치된 버전을 확인합니다. 예상: 2.3.0
from importlib.metadata import version
from mcp.server import MCPServer
from mcp import Client
from mcp.types import TextResourceContents

print("mcp SDK 버전:", version("mcp"))
assert version("mcp") == "2.3.0", "강의 실습 버전을 확인하세요."

mcp SDK 버전: 2.3.0


## 1. 게임 상태 만들기

슬라임과 싸우는 간단한 게임을 구현합니다. **게임 상태는 Python 코드가 보관**합니다. MCP Tool은 이 상태를 조회하거나 변경하는 서버 함수를 외부에 제공합니다.

| 항목 | 시작 값 | 규칙 |
|---|---:|---|
| 플레이어 HP | 100 | 0이면 패배 |
| 슬라임 HP | 60 | 0이면 승리 |
| 포션 | 2개 | 사용 시 HP 30 회복, 최대 100 |
| 공격 | 20 피해 | 공격 1회마다 턴 1 증가 |
| 최대 턴 | 10 | 미종료 시 강제 종료 |

> 단순화를 위해 **몬스터 반격은 구현하지 않습니다**. 실제 대전 규칙 대신 MCP의 호출 흐름에 집중하기 위한 교육용 규칙입니다.

In [3]:
from dataclasses import dataclass, asdict
from typing import Any
import json

@dataclass
class GameState:
    # 서버가 보유하는 게임의 원본 상태입니다.
    player_hp: int = 100
    enemy_hp: int = 60
    potions: int = 2
    turn: int = 0
    outcome: str = "playing"  # playing / win / lose / timeout

# 하나의 게임 인스턴스를 메모리에 생성합니다.
game = GameState()

def reset_game_logic() -> dict[str, Any]:
    """게임을 새로 시작합니다. 테스트 사이의 상태 간섭을 방지할 때 사용합니다."""
    global game
    game = GameState()
    return {"ok": True, "state": asdict(game)}

def status_logic() -> dict[str, Any]:
    """현재 상태만 조회합니다. 이 함수는 턴이나 HP를 바꾸지 않습니다."""
    return {"ok": True, "state": asdict(game)}

def attack_logic(target: str) -> dict[str, Any]:
    """서버가 인자와 게임 규칙을 검증한 뒤에만 공격을 실행합니다."""
    if target != "slime":
        return {"ok": False, "error_code": "INVALID_TARGET", "allowed_targets": ["slime"]}
    if game.outcome != "playing":
        return {"ok": False, "error_code": "GAME_OVER", "outcome": game.outcome}

    # 실제 상태 변경: 모델의 요청이 아니라 서버의 Python 코드가 수행합니다.
    game.enemy_hp = max(0, game.enemy_hp - 20)
    game.turn += 1
    if game.enemy_hp == 0:
        game.outcome = "win"
    elif game.turn >= 10:
        game.outcome = "timeout"
    return {"ok": True, "action": "attack", "state": asdict(game)}

def heal_logic() -> dict[str, Any]:
    """포션이 있고 HP가 부족할 때만 회복합니다."""
    if game.outcome != "playing":
        return {"ok": False, "error_code": "GAME_OVER"}
    if game.potions <= 0:
        return {"ok": False, "error_code": "NO_POTION", "potions": 0}
    if game.player_hp >= 100:
        return {"ok": False, "error_code": "HP_ALREADY_FULL"}
    game.player_hp = min(100, game.player_hp + 30)
    game.potions -= 1
    game.turn += 1
    if game.turn >= 10:
        game.outcome = "timeout"
    return {"ok": True, "action": "heal", "state": asdict(game)}

print("초기 게임 상태:", status_logic()["state"])

초기 게임 상태: {'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'outcome': 'playing'}


## 2. MCP Server 등록: Tool

- **Tool:** 모델 또는 Host가 필요할 때 호출할 수 있는 함수입니다.
- `name`은 Python 함수명에서, `description`은 docstring에서, 입력 스키마는 타입 힌트에서 만들어집니다.
- MCP 서버가 도구를 제공한다고 해서 **모델이 직접 Python 함수를 실행하는 것은 아닙니다**. MCP Client가 프로토콜 메시지를 보내고 서버 코드가 실행합니다.

읽기 전용인 `get_game_status`와 상태를 변경하는 `attack_enemy`·`heal_player`를 분리합니다.

In [4]:
# RPG 기능을 공개할 MCP 서버 객체를 만듭니다.
mcp = MCPServer("RPG Game Server")

@mcp.tool()
def get_game_status() -> dict[str, Any]:
    """현재 플레이어·몬스터 HP, 포션 수와 턴을 조회한다. 게임 상태는 바꾸지 않는다."""
    return status_logic()

@mcp.tool()
def attack_enemy(target: str) -> dict[str, Any]:
    """지정한 몬스터를 공격한다. 현재 지원되는 target은 'slime'이다."""
    return attack_logic(target)

@mcp.tool()
def heal_player() -> dict[str, Any]:
    """포션 한 개를 사용해 플레이어 HP를 최대 30 회복한다."""
    return heal_logic()

@mcp.tool()
def reset_game() -> dict[str, Any]:
    """게임을 초기 상태로 재설정한다. 기존 진행 상태를 덮어쓰는 쓰기 도구다."""
    return reset_game_logic()

print("MCP Server 등록 완료: Tool 4개")

MCP Server 등록 완료: Tool 4개


## 3. MCP Server 등록: Resource와 Prompt

**Resource**는 URI로 읽는 데이터입니다. 이 예제의 몬스터 설명은 게임 상태와 관계없는 **고정된 설정 문서**입니다.  
**Prompt**는 Host/사용자가 불러다 사용할 수 있는 **재사용 가능한 지시문 템플릿**입니다. 템플릿을 가져온 것만으로 LLM이 실행되지는 않습니다.

- Resource: `game://monsters/slime`
- Prompt: `battle_strategy(style)`

In [5]:
@mcp.resource("game://monsters/slime")
def slime_info() -> str:
    """슬라임의 기본 설정을 알려준다. 현재 전투 HP가 아닌 몬스터 도감 정보다."""
    return "슬라임: 초기 HP 60, 플레이어 공격 1회당 피해 20. 세 번 맞으면 쓰러진다."

@mcp.prompt()
def battle_strategy(style: str = "safe") -> str:
    """원하는 전투 스타일에 맞춰 전략을 요청하는 프롬프트 템플릿."""
    return (
        f"당신은 RPG 전투 전략 조언자다. 스타일: {style}. "
        "전투 상태를 먼저 확인하고, 포션과 턴 제한을 고려해 다음 행동을 짧게 제안하라. "
        "도구 결과를 확인하지 않은 행동을 완료했다고 주장하지 마라."
    )

print("등록 완료: Resource 1개 + Prompt 1개")

등록 완료: Resource 1개 + Prompt 1개


## 4. In-memory MCP Client 연결 및 도구 목록 확인

`Client(mcp)`는 서버 객체와 같은 Python 프로세스에서 통신합니다. **HTTP 주소·포트·서버 파일이 필요 없습니다.**

실습 1에서 작성한 수동 함수 목록과 달리, 이번에는 **MCP의 도구 발견 기능**으로 서버가 제공하는 이름과 스키마를 얻습니다.

In [6]:
# Colab/Jupyter는 최상위 await를 지원합니다.
# mcp 객체를 직접 넘겨 테스트용 In-memory 연결을 만듭니다.
async with Client(mcp) as client:
    # 서버에서 공개한 Tool 목록을 조회합니다.
    tool_listing = await client.list_tools()
    for tool in tool_listing.tools:
        print(f"이름: {tool.name}")
        print(f"설명: {tool.description}")
        print("입력 스키마:", json.dumps(tool.input_schema, ensure_ascii=False))
        print("-" * 55)

    tool_names = {tool.name for tool in tool_listing.tools}
    assert {"attack_enemy", "heal_player", "get_game_status", "reset_game"} <= tool_names

print("검증 성공: MCP Tool 4개 검색")

이름: get_game_status
설명: 현재 플레이어·몬스터 HP, 포션 수와 턴을 조회한다. 게임 상태는 바꾸지 않는다.
입력 스키마: {"type": "object", "properties": {}, "title": "get_game_statusArguments"}
-------------------------------------------------------
이름: attack_enemy
설명: 지정한 몬스터를 공격한다. 현재 지원되는 target은 'slime'이다.
입력 스키마: {"type": "object", "properties": {"target": {"title": "Target", "type": "string"}}, "required": ["target"], "title": "attack_enemyArguments"}
-------------------------------------------------------
이름: heal_player
설명: 포션 한 개를 사용해 플레이어 HP를 최대 30 회복한다.
입력 스키마: {"type": "object", "properties": {}, "title": "heal_playerArguments"}
-------------------------------------------------------
이름: reset_game
설명: 게임을 초기 상태로 재설정한다. 기존 진행 상태를 덮어쓰는 쓰기 도구다.
입력 스키마: {"type": "object", "properties": {}, "title": "reset_gameArguments"}
-------------------------------------------------------
검증 성공: MCP Tool 4개 검색


## 5. MCP Client로 전투 실행하기

한 번의 요청에서 데이터가 이동하는 경로는 **Client → tools/call → Server의 Python 함수 → 결과 → Client**입니다.

결과의 두 필드를 구분하세요.
- `is_error`: MCP 도구 실행 자체의 오류 여부
- `structured_content`: 코드에서 처리할 수 있는 구조화 결과

**중요:** 예제의 `ok: False`는 예상 가능한 **게임 업무 오류**이므로, MCP 호출 자체가 실패했다는 뜻은 아닙니다.

In [7]:
async with Client(mcp) as client:
    # 결과가 이전 실행에 의존하지 않도록 먼저 초기화합니다.
    reset_result = await client.call_tool("reset_game", {})
    assert not reset_result.is_error

    # ① 상태 조회: 읽기 작업이므로 턴이 증가하지 않아야 합니다.
    before = await client.call_tool("get_game_status", {})
    print("공격 전:", before.structured_content)
    assert before.structured_content["state"]["enemy_hp"] == 60

    # ② 공격 도구 호출: 슬라임 HP가 60 -> 40이 됩니다.
    attacked = await client.call_tool("attack_enemy", {"target": "slime"})
    print("공격 후:", attacked.structured_content)
    assert attacked.is_error is False
    assert attacked.structured_content["state"]["enemy_hp"] == 40

    # ③ 상태 재조회: 서버 메모리의 HP 변경이 유지되는지 검사합니다.
    after = await client.call_tool("get_game_status", {})
    assert after.structured_content["state"]["enemy_hp"] == 40
    assert after.structured_content["state"]["turn"] == 1

print("검증 성공: 공격이 실제 게임 상태를 변경함")

공격 전: {'ok': True, 'state': {'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'outcome': 'playing'}}
공격 후: {'ok': True, 'action': 'attack', 'state': {'player_hp': 100, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'outcome': 'playing'}}
검증 성공: 공격이 실제 게임 상태를 변경함


## 6. Resource 읽기 · Prompt 가져오기

이 두 기능은 Tool 호출과 구분됩니다.

- `read_resource(uri)`는 **지정된 URI의 내용을 읽습니다**.
- `get_prompt(name, arguments)`는 **문장을 만들어 돌려줍니다**. 전략을 실행하거나 적을 공격하지는 않습니다.

In [8]:
async with Client(mcp) as client:
    # 서버가 제공하는 고정 URI 목록을 확인합니다.
    resources = await client.list_resources()
    print("Resource 목록:", [str(item.uri) for item in resources.resources])

    # 하나의 Resource를 읽고 텍스트를 확인합니다.
    resource_result = await client.read_resource("game://monsters/slime")
    resource_texts = [part.text for part in resource_result.contents
                      if isinstance(part, TextResourceContents)]
    print("슬라임 도감:", resource_texts)
    assert any("슬라임" in text for text in resource_texts)

    # Prompt 목록에서 이름을 찾고 style 인자를 적용합니다.
    prompts = await client.list_prompts()
    print("Prompt 목록:", [item.name for item in prompts.prompts])
    prompt_result = await client.get_prompt("battle_strategy", {"style": "aggressive"})
    print("생성된 프롬프트:")
    for message in prompt_result.messages:
        if hasattr(message.content, "text"):
            print(message.role, ":", message.content.text)

    assert "battle_strategy" in [item.name for item in prompts.prompts]

print("검증 성공: Resource와 Prompt 조회")

Resource 목록: ['game://monsters/slime']
슬라임 도감: ['슬라임: 초기 HP 60, 플레이어 공격 1회당 피해 20. 세 번 맞으면 쓰러진다.']
Prompt 목록: ['battle_strategy']
생성된 프롬프트:
user : 당신은 RPG 전투 전략 조언자다. 스타일: aggressive. 전투 상태를 먼저 확인하고, 포션과 턴 제한을 고려해 다음 행동을 짧게 제안하라. 도구 결과를 확인하지 않은 행동을 완료했다고 주장하지 마라.
검증 성공: Resource와 Prompt 조회


## 7. 실패 상황 테스트: 업무 오류와 입력 오류

**업무 오류:** 대상 몬스터가 없거나 포션이 부족하면 `ok: False`와 `error_code`를 반환합니다.  
**입력 오류:** `target` 인자를 누락하면 SDK의 입력 검증에서 거부될 수 있습니다. 이때 구체적인 오류 표현은 SDK 설정에 따라 `is_error` 결과 또는 예외가 될 수 있으므로 양쪽을 확인합니다.

> 보안 주의: 서버가 스스로 작성한 설명/Annotation은 강제 권한 정책이 아닙니다. 실제 서비스에서는 Host 및 실행 환경의 승인·권한 정책을 따로 구성해야 합니다.

In [9]:
async with Client(mcp) as client:
    await client.call_tool("reset_game", {})

    # ① 허용되지 않는 몬스터는 서버의 업무 규칙으로 거부됩니다.
    wrong_target = await client.call_tool("attack_enemy", {"target": "dragon"})
    print("잘못된 대상:", wrong_target.structured_content)
    assert wrong_target.structured_content["error_code"] == "INVALID_TARGET"
    assert wrong_target.structured_content["ok"] is False
    assert wrong_target.is_error is False  # 정상적인 MCP 호출이지만 업무 처리 실패

    # ② 최대 HP에서 회복하면 포션을 소비하지 않고 업무 오류를 반환합니다.
    full_hp = await client.call_tool("heal_player", {})
    print("최대 HP에서 회복:", full_hp.structured_content)
    assert full_hp.structured_content["error_code"] == "HP_ALREADY_FULL"

    # ③ 필수 인자 누락은 SDK가 거부해야 합니다. 반환 방식은 환경별로 확인합니다.
    try:
        missing = await client.call_tool("attack_enemy", {})
        print("필수 인자 누락: is_error =", missing.is_error)
        print("메시지 =", missing.content)
        assert missing.is_error is True
    except Exception as exc:
        print("필수 인자 누락: SDK 예외 =", type(exc).__name__, str(exc)[:200])

    # 잘못된 호출 후 게임 상태가 바뀌지 않아야 합니다.
    current = await client.call_tool("get_game_status", {})
    assert current.structured_content["state"]["enemy_hp"] == 60
    assert current.structured_content["state"]["turn"] == 0

print("검증 성공: 잘못된 입력에서 게임 상태가 변경되지 않음")

잘못된 대상: {'ok': False, 'error_code': 'INVALID_TARGET', 'allowed_targets': ['slime']}
최대 HP에서 회복: {'ok': False, 'error_code': 'HP_ALREADY_FULL'}
필수 인자 누락: is_error = True
메시지 = [TextContent(type='text', text='Error executing tool attack_enemy: 1 validation error for attack_enemyArguments\ntarget\n  Field required [type=missing, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.14/v/missing', annotations=None, meta=None)]
검증 성공: 잘못된 입력에서 게임 상태가 변경되지 않음


## 8. 자동 Contract Test: 정상·경계·반복·오류

다음 셀은 테스트를 한 번에 수행합니다. 테스트 결과가 성공하면 `총 7개 체크 통과`가 출력됩니다.  
서버 상태가 테스트끼리 영향을 주지 않도록 각 시나리오 시작 시 `reset_game()`을 호출합니다.

In [10]:
async def run_contract_tests() -> None:
    passed = 0

    async with Client(mcp) as client:
        # 1) 초기 상태
        initial = await client.call_tool("reset_game", {})
        assert initial.structured_content["state"]["enemy_hp"] == 60
        passed += 1

        # 2) 읽기 도구는 상태를 변경하지 않음
        first = await client.call_tool("get_game_status", {})
        second = await client.call_tool("get_game_status", {})
        assert first.structured_content["state"] == second.structured_content["state"]
        passed += 1

        # 3) 공격 한 번으로 HP 감소 및 턴 증가
        attack = await client.call_tool("attack_enemy", {"target": "slime"})
        assert (attack.structured_content["state"]["enemy_hp"],
                attack.structured_content["state"]["turn"]) == (40, 1)
        passed += 1

        # 4) 잘못된 target에는 상태 변경이 없음
        invalid = await client.call_tool("attack_enemy", {"target": "dragon"})
        assert invalid.structured_content["error_code"] == "INVALID_TARGET"
        state = await client.call_tool("get_game_status", {})
        assert state.structured_content["state"]["enemy_hp"] == 40
        passed += 1

        # 5) HP가 100이면 포션을 낭비하지 않음
        heal = await client.call_tool("heal_player", {})
        assert heal.structured_content["error_code"] == "HP_ALREADY_FULL"
        state = await client.call_tool("get_game_status", {})
        assert state.structured_content["state"]["potions"] == 2
        passed += 1

        # 6) 총 3번 공격하면 승리
        await client.call_tool("attack_enemy", {"target": "slime"})
        win = await client.call_tool("attack_enemy", {"target": "slime"})
        assert win.structured_content["state"]["enemy_hp"] == 0
        assert win.structured_content["state"]["outcome"] == "win"
        passed += 1

        # 7) 게임 종료 후에는 추가 공격이 허용되지 않음
        extra = await client.call_tool("attack_enemy", {"target": "slime"})
        assert extra.structured_content["error_code"] == "GAME_OVER"
        passed += 1

    print(f"✅ 총 {passed}개 체크 통과")

await run_contract_tests()

✅ 총 7개 체크 통과


## 9. 학습 정리: MCP가 맡는 역할

| 구분 | 이 실습에서 확인한 역할 |
|---|---|
| **Tool** | 함수 목록을 공개하고 `tools/call`로 실제 실행 |
| **Resource** | 고정 URI의 몬스터 도감 데이터를 조회 |
| **Prompt** | 전투 전략 지시문 템플릿을 반환 (실행하지 않음) |
| **Client** | 서버의 도구를 발견하고 요청·결과를 주고받음 |
| **Server** | 도구를 제공하며 Python 함수로 게임 상태를 변경함 |
| **Host** | 실제 앱에서는 사용자 상호작용, 서버 신뢰 및 승인을 담당함. 이 실습은 Host UI를 구현하지 않음 |

**핵심:** 실습 1의 Python 함수를 MCP 표준 인터페이스로 제공했습니다. **MCP는 연결 방식의 표준이며, 모델의 판단·권한 검사를 대신하지 않습니다.** 다음 실습 3에서는 이 도구들을 LangGraph의 노드에서 호출하도록 확장합니다.

### 참고 문서
- MCP Python SDK Client: https://py.sdk.modelcontextprotocol.io/client/
- MCP Python SDK Server: https://py.sdk.modelcontextprotocol.io/server/
- 강의자료: 9장 Tool Calling · MCP · LangGraph (MCP 서버 구현 및 In-memory Client)

**Colab 실행이 실패한다면:** 설치된 `mcp` 버전 확인 → 위에서부터 재실행 → 오류가 난 셀과 오류 메시지를 함께 확인하세요.